# **The Growing Cost of Care**

## <u>1 - Project Overview</u>
This project investigates the relationship between patient age,
medication utilization, insurance coverage, and uncovered
healthcare costs using the Synthea synthetic healthcare dataset.

#### 1.1 - Central Research Question
As patients age and take more medications, does insurance
coverage keep pace, or do patients bear a growing share of costs?

#### 1.2 - Project Objectives
1. Examine medication utilization across patient age groups.
2. Investigate insurance coverage across age groups and payers.
3. Analyze uncovered medication costs by age and medication count.

#### 1.3 - Technology Stack
- Microsoft Fabric Lakehouse
- SQL (Spark SQL)
- Python (PySpark)
- Delta Lake
- Power BI

#### 1.4 - Data Architecture
- Bronze: Original source tables accessed through Lakehouse shortcuts.
- Silver: Cleaned and validated datasets.
- Gold: Analysis-ready fact tables, dimension tables, and aggregates.
- Visualization: Power BI dashboard.

#### 1.5 - Data Source
Synthea synthetic healthcare dataset, loaded into a Micrsoft 
Fabric Lakehouse.

#### 1.6 - Important Data Considerations
- Source tables will remain unchanged.
- Patient identifiers will be used for joins.
- Direct personal identifiers will be excluded from analytical outputs.
- Findings will describe associations, not establish causation.


## <u>2 - Source Data Exploration</u>

#### 2.1 Source Table Record Counts

In [3]:
%%sql
-- ============================================================
-- QUERY 1: RECORD COUNTS
-- ============================================================

-- Count the total records in the patients table.
-- This establishes the size of the patient population.
SELECT
    COUNT(*) AS total_patient_records
FROM dbo_1.patients;

-- Count the total records in the medications table.
-- Each row represents a medication record, not necessarily
-- a distinct medication or a distinct patient.
SELECT
    COUNT(*) AS total_medication_records
FROM dbo_1.medications;

-- Count the number of insurance payer records.
-- This table will be later used to identify payer names.
SELECT
    COUNT(*) AS total_payer_records
FROM dbo_1.payers;


StatementMeta(, a54bb5ee-e36c-4e77-960a-b15d7d8f4230, 7, Finished, Available, Finished, True)

<Spark SQL result set with 1 rows and 1 fields>

<Spark SQL result set with 1 rows and 1 fields>

<Spark SQL result set with 1 rows and 1 fields>

#### 2.2 Source Data Range

In [4]:
%%sql
-- ============================================================
-- QUERY 2: DATE RANGE EXPLORATION
-- ============================================================

-- Examine the earliest and latest patient birthdates.
-- This helps to understand the demographic range of the dataset.
SELECT
    MIN(BIRTHDATE) AS earliest_birthdate,
    MAX(BIRTHDATE) AS latest_birthdate
FROM dbo_1.patients;

-- Examine the medication start and stop dates.
-- This shows the historical period covered by the medication data.
SELECT
    MIN(START) AS earliest_medication_start,
    MAX(START) AS latest_medication_start,
    MIN(STOP) AS earliest_medication_stop,
    MAX(STOP) AS latest_medication_stop
FROM dbo_1.medications;


StatementMeta(, a54bb5ee-e36c-4e77-960a-b15d7d8f4230, 9, Finished, Available, Finished, True)

<Spark SQL result set with 1 rows and 2 fields>

<Spark SQL result set with 1 rows and 4 fields>

#### 2.3 - Missing Value Assessment

In [5]:
%%sql
-- ============================================================
-- QUERY 3: MISSING VALUE ASSESSMENT
-- ============================================================

-- Check missing values in the patient identifier and birthdate.
-- These fields are important for linking patients and calculating age.
SELECT
    COUNT(*) AS total_records,
    SUM(CASE WHEN Id IS NULL THEN 1 ELSE 0 END) AS missing_patient_ids,
    SUM(CASE WHEN BIRTHDATE IS NULL THEN 1 ELSE 0 END) AS missing_birthdates
FROM dbo_1.patients;

-- Check missing values in the medication fields.
-- These fields are necessary for patient joins, date calculations,
-- payer analysis, and medication cost calculations.
SELECT
    COUNT(*) AS total_records,
    SUM(CASE WHEN PATIENT IS NULL THEN 1 ELSE 0 END) AS missing_patient_ids,
    SUM(CASE WHEN START IS NULL THEN 1 ELSE 0 END) AS missing_start_dates,
    SUM(CASE WHEN PAYER IS NULL THEN 1 ELSE 0 END) AS missing_payer_ids,
    SUM(CASE WHEN TOTALCOST IS NULL THEN 1 ELSE 0 END) AS missing_total_costs,
    SUM(CASE WHEN PAYER_COVERAGE IS NULL THEN 1 ELSE 0 END) AS missing_payer_coverage
FROM dbo_1.medications;


StatementMeta(, a54bb5ee-e36c-4e77-960a-b15d7d8f4230, 11, Finished, Available, Finished, True)

<Spark SQL result set with 1 rows and 3 fields>

<Spark SQL result set with 1 rows and 6 fields>

#### 2.4 Patient-to-Medication Relationship

In [7]:

# ============================================================
# QUERY 4: PATIENT-TO-MEDICATION RELATIONSHIP
# ============================================================

# STEP 1: IMPORT PYSPARK FUNCTIONS
# Import the functions library and assign it the alias F.
# We use F for operations such as selecting columns,
# filtering records, counting values, and performing calculations.
from pyspark.sql import functions as F


# STEP 2: LOAD SOURCE TABLES
# Read the original patients table from our Lakehouse.
# This creates a Spark DataFrame named patients.
patients = spark.table("dbo_1.patients")

# Read the original medications table from our Lakehouse.
# This creates a Spark DataFrame named medications.
medications = spark.table("dbo_1.medications")


# STEP 3: PREPARE PATIENT IDENTIFIERS
# Select the patient ID column from the patients table.
# Rename Id to PATIENT so it matches the identifier column
# in the medications table.
patient_ids = patients.select(
    F.col("Id").alias("PATIENT")
)


# STEP 4: IDENTIFY UNMATCHED MEDICATION RECORDS
# A left anti join returns rows from medications that
# do not have a corresponding patient ID in patients.
#
# PATIENT is the common column used to match the two tables.
unmatched_medications = medications.join(
    patient_ids,
    on="PATIENT",
    how="left_anti"
)


# STEP 5: COUNT UNMATCHED RECORDS
# Count how many medication records do not have
# a matching patient in the patients table.
unmatched_count = unmatched_medications.count()


# STEP 6: DISPLAY THE RESULT
# A result of zero means every medication record
# has a matching patient ID in our patient table.
print("Medication records without a matching patient:", unmatched_count)


StatementMeta(, a54bb5ee-e36c-4e77-960a-b15d7d8f4230, 13, Finished, Available, Finished, False)

Medication records without a matching patient: 0


#### 2.5 Medication Cost and Coverage Profile

In [8]:

# ============================================================
# QUERY 5: MEDICATION COSTS AND INSURANCE COVERAGE
# ============================================================

# STEP 1: IMPORT PYSPARK FUNCTIONS
# Import the functions library for our financial calculations.
from pyspark.sql import functions as F


# STEP 2: LOAD THE MEDICATION TABLE
# Read the original medications table from our Lakehouse.
medications = spark.table("dbo_1.medications")


# STEP 3: CALCULATE FINANCIAL SUMMARY STATISTICS
# agg() allows us to calculate several summary statistics
# across the entire medications DataFrame.
financial_profile = medications.agg(

    # Calculate the total cost across all medication records.
    F.sum("TOTALCOST").alias("total_medication_cost"),

    # Calculate the total amount covered by insurance payers.
    F.sum("PAYER_COVERAGE").alias("total_payer_coverage"),

    # Calculate preliminary uncovered costs.
    # This subtracts payer coverage from total medication cost.
    F.sum(
        F.col("TOTALCOST") - F.col("PAYER_COVERAGE")
    ).alias("calculated_uncovered_cost"),

    # Count records where insurance coverage exceeds total cost.
    # Such records need investigation before financial analysis.
    F.sum(
        F.when(
            F.col("PAYER_COVERAGE") > F.col("TOTALCOST"), 1
        ).otherwise(0)
    ).alias("coverage_exceeds_total_cost"),

    # Count records with negative total medication costs.
    # These may represent unusual records or potential data issues.
    F.sum(
        F.when(F.col("TOTALCOST") < 0, 1).otherwise(0)
    ).alias("negative_total_cost_records")

)


# STEP 4: DISPLAY THE FINANCIAL SUMMARY
# Show the calculated results in a readable table.
financial_profile.show(truncate=False)


StatementMeta(, a54bb5ee-e36c-4e77-960a-b15d7d8f4230, 14, Finished, Available, Finished, False)

+---------------------+--------------------+-------------------------+---------------------------+---------------------------+
|total_medication_cost|total_payer_coverage|calculated_uncovered_cost|coverage_exceeds_total_cost|negative_total_cost_records|
+---------------------+--------------------+-------------------------+---------------------------+---------------------------+
|1.1187020335918919E10|4.1481197367223537E8|1.0772208362275362E10    |0                          |0                          |
+---------------------+--------------------+-------------------------+---------------------------+---------------------------+



#### 2.6 Duplicate Patient Identifier Check

In [9]:
%%sql
-- ============================================================
-- QUERY 6: DUPLICATE PATIENT IDENTIFIERS
-- ============================================================

-- Group patient records by their ID.
-- If an ID appears more than once, it may indicate duplicate
-- patient records that need further investigation.
SELECT
    Id,
    COUNT(*) AS id_count
FROM dbo_1.patients
GROUP BY Id
HAVING COUNT(*) > 1
ORDER BY id_count DESC;

StatementMeta(, a54bb5ee-e36c-4e77-960a-b15d7d8f4230, 15, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 2 fields>

#### 2.7 Payer Referential Integrity

In [10]:
%%sql
-- ============================================================
-- QUERY 7: PAYER REFERENTIAL INTEGRITY (Checking whether medication payers IDs match the payers table)
-- ============================================================

-- Match the PAYER column in medications to the Id column
-- in payers using a LEFT JOIN.
--
-- A LEFT JOIN retains every medication record, even if
-- its payer ID cannot be found in the payers table.
--
-- We then count medication records without a matching payer.
SELECT
    COUNT(*) AS unmatched_payer_records
FROM dbo_1.medications AS m
LEFT JOIN dbo_1.payers AS p
    ON m.PAYER = p.Id
WHERE p.Id IS NULL;


StatementMeta(, a54bb5ee-e36c-4e77-960a-b15d7d8f4230, 16, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

#### 2.8 Base Cost and Total Cost Validation

In [11]:
%%sql
-- ============================================================
-- QUERY 8: BASE COST AND TOTAL COST VALIDATION
-- ============================================================

-- Compare the reported TOTALCOST with the calculated value
-- BASE_COST multiplied by DISPENSES.
--
-- We use a tolerance of 0.01 to avoid treating tiny
-- floating-point differences as meaningful discrepancies.
SELECT
    COUNT(*) AS total_records,

    SUM(
        CASE
            WHEN ABS(
                TOTALCOST - (BASE_COST * DISPENSES)
            ) > 0.01
            THEN 1
            ELSE 0
        END
    ) AS records_with_cost_difference,

    SUM(
        CASE
            WHEN BASE_COST IS NULL
              OR DISPENSES IS NULL
              OR TOTALCOST IS NULL
            THEN 1
            ELSE 0
        END
    ) AS records_with_missing_cost_inputs

FROM dbo_1.medications;


StatementMeta(, a54bb5ee-e36c-4e77-960a-b15d7d8f4230, 17, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 3 fields>

#### 2.9 Financial Summary by Payer

In [12]:
%%sql
-- ============================================================
-- QUERY 9: MEDICATION COSTS BY INSURANCE PAYER
-- ============================================================

-- Join medications to payers using their shared payer identifier.
-- This allows us to group financial information by payer name.

SELECT
    p.NAME AS payer_name,

    -- Count the number of medication records associated with each payer.
    COUNT(*) AS medication_records,

    -- Calculate the total medication cost for each payer.
    SUM(m.TOTALCOST) AS total_medication_cost,

    -- Calculate how much each payer covered.
    SUM(m.PAYER_COVERAGE) AS total_payer_coverage,

    -- Calculate the difference between medication cost and coverage.
    SUM(m.TOTALCOST - m.PAYER_COVERAGE) AS calculated_uncovered_cost,

    -- Calculate the weighted coverage percentage.
    -- We divide total coverage by total cost, rather than averaging
    -- individual medication-record coverage percentages.
    ROUND(
        100.0 * SUM(m.PAYER_COVERAGE)
        / NULLIF(SUM(m.TOTALCOST), 0),
        2
    ) AS coverage_percentage

FROM dbo_1.medications AS m

INNER JOIN dbo_1.payers AS p
    ON m.PAYER = p.Id

-- Group the results by insurance provider.
GROUP BY p.NAME

-- Show the largest medication cost totals first.
ORDER BY total_medication_cost DESC;


StatementMeta(, a54bb5ee-e36c-4e77-960a-b15d7d8f4230, 18, Finished, Available, Finished, False)

<Spark SQL result set with 10 rows and 6 fields>

#### 2.10 Coverage Distribution by Payer

In [13]:
%%sql
-- ============================================================
-- QUERY 10: COVERAGE STATUS BY PAYER
-- ============================================================

-- Categorize medication records based on whether the payer
-- contributed any coverage.
-- This is a preliminary classification, not a determination
-- of the patient's actual insurance status.

SELECT
    p.NAME AS payer_name,

    -- Count all medication records associated with the payer.
    COUNT(*) AS total_medication_records,

    -- Count records with positive payer coverage.
    SUM(
        CASE
            WHEN m.PAYER_COVERAGE > 0 THEN 1
            ELSE 0
        END
    ) AS records_with_positive_coverage,

    -- Count records with zero payer coverage.
    SUM(
        CASE
            WHEN m.PAYER_COVERAGE = 0 THEN 1
            ELSE 0
        END
    ) AS records_with_zero_coverage,

    -- Calculate the average amount covered per medication record.
    AVG(m.PAYER_COVERAGE) AS average_payer_coverage,

    -- Calculate the average total medication cost per record.
    AVG(m.TOTALCOST) AS average_medication_cost

FROM dbo_1.medications AS m

INNER JOIN dbo_1.payers AS p
    ON m.PAYER = p.Id

-- Group all records by payer.
GROUP BY p.NAME

-- Sort alphabetically by payer name.
ORDER BY payer_name;


StatementMeta(, a54bb5ee-e36c-4e77-960a-b15d7d8f4230, 19, Finished, Available, Finished, False)

<Spark SQL result set with 10 rows and 6 fields>

In [14]:
%%sql
-- Query 11: Examine coverage only among medication records
-- that have a positive amount of payer coverage to see if 
-- low coverage percentages are caused primarily by records
-- with zero coverage or by low coverage amounts on records 
-- that do receive coverage.

SELECT
    -- Identify the insurance payer.
    p.NAME AS payer_name,

    -- Count all medication records for this payer.
    COUNT(*) AS total_medication_records,

    -- Count records where the payer contributed something.
    SUM(
        CASE
            WHEN m.PAYER_COVERAGE > 0 THEN 1
            ELSE 0
        END
    ) AS records_with_positive_coverage,

    -- Calculate the percentage of records receiving coverage.
    ROUND(
        100.0 * SUM(
            CASE
                WHEN m.PAYER_COVERAGE > 0 THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS percent_records_with_coverage,

    -- Calculate the average coverage amount,
    -- but only among records with positive coverage.
    ROUND(
        AVG(
            CASE
                WHEN m.PAYER_COVERAGE > 0
                THEN m.PAYER_COVERAGE
            END
        ),
        2
    ) AS avg_coverage_when_positive,

    -- Calculate the average medication cost
    -- among records with positive coverage.
    ROUND(
        AVG(
            CASE
                WHEN m.PAYER_COVERAGE > 0
                THEN m.TOTALCOST
            END
        ),
        2
    ) AS avg_cost_when_covered,

    -- Calculate the weighted coverage percentage
    -- using only records with positive coverage.
    ROUND(
        100.0 * SUM(
            CASE
                WHEN m.PAYER_COVERAGE > 0
                THEN m.PAYER_COVERAGE
                ELSE 0
            END
        ) / NULLIF(
            SUM(
                CASE
                    WHEN m.PAYER_COVERAGE > 0
                    THEN m.TOTALCOST
                    ELSE 0
                END
            ),
            0
        ),
        2
    ) AS coverage_pct_when_positive

-- Start with medication records.
FROM dbo_1.medications AS m

-- Match each medication record to its payer.
INNER JOIN dbo_1.payers AS p
    ON m.PAYER = p.Id

-- Produce one row per payer.
GROUP BY p.NAME

-- Display the payers alphabetically.
ORDER BY payer_name;

StatementMeta(, a54bb5ee-e36c-4e77-960a-b15d7d8f4230, 20, Finished, Available, Finished, False)

<Spark SQL result set with 10 rows and 7 fields>

## <u>3 - Silver Layer Medication Data Cleaning and Transformation</u>

#### 3.1 Creating silver medication table

In [15]:
# STEP 1: Import PySpark functions.
# F gives us access to functions such as year(), when(),
# col(), lit(), and add_months().
from pyspark.sql import functions as F


# STEP 2: Load the original source tables from the instructor's
# Lakehouse schema.
# We are reading these tables only; we will not modify them.

patients_df = spark.table("dbo_1.patients")
medications_df = spark.table("dbo_1.medications")


# STEP 3: Select the patient fields needed for our analysis.
# Rename columns into consistent, lowercase snake_case names.
# We intentionally exclude direct personal identifiers.

patients_clean = patients_df.select(
    F.col("Id").alias("patient_id"),
    F.col("BIRTHDATE").alias("birth_date"),
    F.col("DEATHDATE").alias("death_date"),
    F.col("GENDER").alias("gender"),
    F.col("RACE").alias("race"),
    F.col("ETHNICITY").alias("ethnicity"),
    F.col("CITY").alias("city"),
    F.col("STATE").alias("state"),
    F.col("COUNTY").alias("county")
)


# STEP 4: Select and rename the medication fields.
# These contain the medication details, dates, payer information,
# and financial values that we need for the project.

medications_clean = medications_df.select(
    F.col("START").alias("medication_start"),
    F.col("STOP").alias("medication_stop"),
    F.col("PATIENT").alias("patient_id"),
    F.col("PAYER").alias("payer_id"),
    F.col("ENCOUNTER").alias("encounter_id"),
    F.col("CODE").alias("medication_code"),
    F.col("DESCRIPTION").alias("medication_description"),
    F.col("BASE_COST").alias("base_cost"),
    F.col("PAYER_COVERAGE").alias("payer_coverage"),
    F.col("TOTALCOST").alias("total_cost"),
    F.col("DISPENSES").alias("dispenses"),
    F.col("REASONCODE").alias("reason_code"),
    F.col("REASONDESCRIPTION").alias("reason_description")
)


# STEP 5: Join medication records with patient demographics.
# patient_id is the common key:
# medications.PATIENT = patients.Id
#
# An INNER JOIN keeps only medication records with a matching
# patient. We already verified that every medication record
# has a matching patient in our source data.

silver_medications = medications_clean.join(
    patients_clean,
    on="patient_id",
    how="inner"
)


# STEP 6: Calculate the patient's age when the medication started.
#
# First, calculate the difference between the calendar years.
# Then check whether the patient's birthday had occurred yet
# in the medication's start year.
#
# If the birthday had not occurred, subtract one year.
# This gives us age at medication start rather than age today.

silver_medications = silver_medications.withColumn(
    "year_difference",
    F.year(F.col("medication_start")) - F.year(F.col("birth_date"))
)

silver_medications = silver_medications.withColumn(
    "age_at_medication_start",
    F.col("year_difference") -
    F.when(
        F.col("medication_start") <
        F.add_months(
            F.col("birth_date"),
            F.col("year_difference") * 12
        ),
        1
    ).otherwise(0)
)


# STEP 7: Assign patients to the age groups from our project outline.
#
# Under 18 is included so that we do not silently discard
# younger patients from the dataset.
#
# The five adult groups match the original project plan.

silver_medications = silver_medications.withColumn(
    "age_group",
    F.when(F.col("age_at_medication_start").isNull(), "Unknown")
     .when(F.col("age_at_medication_start") < 0, "Invalid Age")
     .when(F.col("age_at_medication_start") < 18, "Under 18")
     .when(F.col("age_at_medication_start").between(18, 29), "18-29")
     .when(F.col("age_at_medication_start").between(30, 44), "30-44")
     .when(F.col("age_at_medication_start").between(45, 59), "45-59")
     .when(F.col("age_at_medication_start").between(60, 74), "60-74")
     .when(F.col("age_at_medication_start") >= 75, "75+")
     .otherwise("Unknown")
)


# STEP 8: Calculate the uncovered medication cost.
#
# This is the difference between total medication cost
# and the amount recorded as covered by the payer.
#
# We call this "calculated_uncovered_cost" rather than
# patient_out_of_pocket_cost because we have not verified
# that the difference represents actual patient spending.

silver_medications = silver_medications.withColumn(
    "calculated_uncovered_cost",
    F.col("total_cost") - F.col("payer_coverage")
)


# STEP 9: Remove the temporary year_difference column.
# It was only needed to calculate the patient's age.

silver_medications = silver_medications.drop("year_difference")


# STEP 10: Write the transformed dataset to our own Delta table.
#
# The table is named silver_medications.
# Overwrite allows us to rerun this transformation as we develop
# the pipeline without creating duplicate output tables.
#
# This writes our output; it does NOT overwrite the source tables.

silver_medications.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_medications")


# STEP 11: Display a confirmation and basic information
# about the resulting Silver table.

print("Silver medication table created successfully!")

print(
    "Total medication records:",
    spark.table("silver_medications").count()
)

print("Silver table schema:")
spark.table("silver_medications").printSchema()

# Display a small sample to inspect the transformed records.
display(
    spark.table("silver_medications").limit(10)
)

StatementMeta(, a54bb5ee-e36c-4e77-960a-b15d7d8f4230, 21, Finished, Available, Finished, False)

Silver medication table created successfully!
Total medication records: 4227723
Silver table schema:
root
 |-- patient_id: string (nullable = true)
 |-- medication_start: date (nullable = true)
 |-- medication_stop: date (nullable = true)
 |-- payer_id: string (nullable = true)
 |-- encounter_id: string (nullable = true)
 |-- medication_code: integer (nullable = true)
 |-- medication_description: string (nullable = true)
 |-- base_cost: double (nullable = true)
 |-- payer_coverage: double (nullable = true)
 |-- total_cost: double (nullable = true)
 |-- dispenses: integer (nullable = true)
 |-- reason_code: long (nullable = true)
 |-- reason_description: string (nullable = true)
 |-- birth_date: date (nullable = true)
 |-- death_date: date (nullable = true)
 |-- gender: string (nullable = true)
 |-- race: string (nullable = true)
 |-- ethnicity: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- county: string (nullable = true)
 |-- ag

SynapseWidget(Synapse.DataFrame, 364b5c72-708f-4d8f-89fc-88ff9018b5a6)

#### 3.2 Validate Age Calculations and Age Groups

In [16]:
%%sql
-- ============================================================
-- SILVER VALIDATION 1: CHECK AGE GROUPS
-- ============================================================

-- Summarize the number of medication records in each age group.
-- Also inspect the age range and check for missing ages.

SELECT
    age_group,

    -- Number of medication records assigned to this age group.
    COUNT(*) AS medication_records,

    -- Number of distinct patients represented in this group.
    COUNT(DISTINCT patient_id) AS unique_patients,

    -- Find the youngest and oldest calculated ages.
    MIN(age_at_medication_start) AS minimum_age,
    MAX(age_at_medication_start) AS maximum_age,

    -- Count records where age could not be calculated.
    SUM(
        CASE
            WHEN age_at_medication_start IS NULL THEN 1
            ELSE 0
        END
    ) AS missing_age_records

FROM silver_medications

-- Group the results by age category.
GROUP BY age_group

-- Display groups in a logical order.
ORDER BY
    CASE age_group
        WHEN 'Under 18' THEN 1
        WHEN '18-29' THEN 2
        WHEN '30-44' THEN 3
        WHEN '45-59' THEN 4
        WHEN '60-74' THEN 5
        WHEN '75+' THEN 6
        WHEN 'Invalid Age' THEN 7
        ELSE 8
    END;


StatementMeta(, a54bb5ee-e36c-4e77-960a-b15d7d8f4230, 22, Finished, Available, Finished, False)

<Spark SQL result set with 6 rows and 6 fields>

#### 3.3 Validate Financial Data Quality

In [17]:
%%sql
-- ============================================================
-- SILVER VALIDATION 2: FINANCIAL DATA QUALITY
-- ============================================================

SELECT

    -- Total number of medication records.
    COUNT(*) AS total_medication_records,

    -- Sum of all medication costs.
    ROUND(SUM(total_cost), 2) AS total_medication_cost,

    -- Sum of all recorded payer contributions.
    ROUND(SUM(payer_coverage), 2) AS total_payer_coverage,

    -- Sum of calculated uncovered costs.
    ROUND(SUM(calculated_uncovered_cost), 2)
        AS total_calculated_uncovered_cost,

    -- Check for missing financial values.
    SUM(
        CASE
            WHEN total_cost IS NULL
              OR payer_coverage IS NULL
              OR calculated_uncovered_cost IS NULL
            THEN 1
            ELSE 0
        END
    ) AS records_with_missing_financial_values,

    -- Check for negative medication costs.
    SUM(
        CASE
            WHEN total_cost < 0 THEN 1
            ELSE 0
        END
    ) AS records_with_negative_cost,

    -- Check for negative payer coverage.
    SUM(
        CASE
            WHEN payer_coverage < 0 THEN 1
            ELSE 0
        END
    ) AS records_with_negative_coverage,

    -- Check whether payer coverage exceeds medication cost.
    SUM(
        CASE
            WHEN payer_coverage > total_cost THEN 1
            ELSE 0
        END
    ) AS records_where_coverage_exceeds_cost,

    -- Verify that our calculated uncovered cost is correct.
    SUM(
        CASE
            WHEN ABS(
                calculated_uncovered_cost -
                (total_cost - payer_coverage)
            ) > 0.01
            THEN 1
            ELSE 0
        END
    ) AS records_with_incorrect_uncovered_cost

FROM silver_medications;

StatementMeta(, a54bb5ee-e36c-4e77-960a-b15d7d8f4230, 23, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 9 fields>

## <u>4 - Silver Layer Payer Enrichment</u>

#### 4.1 Create Enriched Silver Medication Table 

In [19]:
# ============================================================
# SILVER LAYER UPDATE: ADD PAYER / INSURANCE COMPANY NAME
# ============================================================

from pyspark.sql import functions as F

# 1. Load our existing Silver medication table.
silver_df = spark.table("silver_medications")

# 2. Load the original payer lookup table.
payers_df = spark.table("dbo_1.payers")

# 3. Keep only the payer ID and readable payer name.
#    Rename the columns so they match our Silver table.
payers_lookup = payers_df.select(
    F.col("Id").alias("payer_id"),
    F.col("NAME").alias("payer_name")
)

# 4. Join the two tables using payer_id.
#    A broadcast join is efficient because the payer table
#    contains only 10 records.
#    A LEFT JOIN preserves every medication record.
silver_enriched = silver_df.join(
    F.broadcast(payers_lookup),
    on="payer_id",
    how="left"
)

# 5. Save the enriched result as a new Delta table.
silver_enriched.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_medications_enriched")

print("Enriched Silver table created successfully!")


StatementMeta(, a54bb5ee-e36c-4e77-960a-b15d7d8f4230, 25, Finished, Available, Finished, False)

Enriched Silver table created successfully!


#### 4.2 Validate Payer Enrichment

In [20]:
# ============================================================
# VALIDATE THE ENRICHED SILVER TABLE
# ============================================================

# Load the newly created table.
enriched_df = spark.table("silver_medications_enriched")

# Check record count and missing payer names.
validation = enriched_df.agg(
    F.count("*").alias("total_medication_records"),

    F.sum(
        F.when(F.col("payer_name").isNull(), 1).otherwise(0)
    ).alias("records_with_missing_payer_names"),

    F.countDistinct("payer_name").alias("unique_payer_names")
)

display(validation)

# Show the number of medication records associated with each payer.
display(
    enriched_df.groupBy("payer_name")
    .count()
    .orderBy(F.desc("count"))
)

# Confirm the updated table structure.
enriched_df.printSchema()


StatementMeta(, a54bb5ee-e36c-4e77-960a-b15d7d8f4230, 26, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 289da108-92cf-4e71-8e06-e12f2bf69d22)

SynapseWidget(Synapse.DataFrame, e2d75d83-bb17-4936-8722-4d776f64099d)

root
 |-- payer_id: string (nullable = true)
 |-- patient_id: string (nullable = true)
 |-- medication_start: date (nullable = true)
 |-- medication_stop: date (nullable = true)
 |-- encounter_id: string (nullable = true)
 |-- medication_code: integer (nullable = true)
 |-- medication_description: string (nullable = true)
 |-- base_cost: double (nullable = true)
 |-- payer_coverage: double (nullable = true)
 |-- total_cost: double (nullable = true)
 |-- dispenses: integer (nullable = true)
 |-- reason_code: long (nullable = true)
 |-- reason_description: string (nullable = true)
 |-- birth_date: date (nullable = true)
 |-- death_date: date (nullable = true)
 |-- gender: string (nullable = true)
 |-- race: string (nullable = true)
 |-- ethnicity: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- county: string (nullable = true)
 |-- age_at_medication_start: integer (nullable = true)
 |-- age_group: string (nullable = true)
 |-- calcul

#### 4.3 Confirm Record Count Preservation

In [21]:
# Confirm that adding payer names did not change our record count
# and that every medication record has a matching payer name.

enriched_df = spark.table("silver_medications_enriched")

print("Total medication records:", enriched_df.count())

print(
    "Records with missing payer names:",
    enriched_df.filter(F.col("payer_name").isNull()).count()
)

print(
    "Unique payer names:",
    enriched_df.select("payer_name").distinct().count()
)


StatementMeta(, a54bb5ee-e36c-4e77-960a-b15d7d8f4230, 27, Finished, Available, Finished, False)

Total medication records: 4227723
Records with missing payer names: 0
Unique payer names: 10


## <u>Section 5 - Gold Layer Dimension Tables</u>

#### 5.1  Create Patient Dimension

In [23]:
# ============================================================
# GOLD LAYER - STEP 1: BUILD DIM_PATIENTS
# Include ALL patients from the original source table (even 
# patients with no medical records because a dimension table 
# should represent the full patient population available in 
# our source data. Even if some patients have no medication 
# records, retaining them gives us a more complete model and 
# allows us to distinguish between patients with and without 
# medication activity.).
# ============================================================

from pyspark.sql import functions as F

# 1. Load the original patients source table.
#    This includes patients who may not have medication records.
patients_df = spark.table("dbo_1.patients")

# 2. Select the patient-level demographic columns we need.
dim_patients = patients_df.select(
    F.col("Id").alias("patient_id"),
    F.col("BIRTHDATE").alias("birth_date"),
    F.col("DEATHDATE").alias("death_date"),
    F.col("GENDER").alias("gender"),
    F.col("RACE").alias("race"),
    F.col("ETHNICITY").alias("ethnicity"),
    F.col("CITY").alias("city"),
    F.col("STATE").alias("state"),
    F.col("COUNTY").alias("county")
)

# 3. Remove duplicate patient IDs, if any.
dim_patients = dim_patients.dropDuplicates(["patient_id"])

# 4. Save the complete patient dimension.
#    This replaces our earlier, smaller version.
dim_patients.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dim_patients")

# 5. Confirm the final patient count.
print("Updated patient dimension created successfully!")
print("Total unique patients:", dim_patients.count())

# 6. Preview the resulting table.
display(dim_patients.limit(10))


StatementMeta(, a54bb5ee-e36c-4e77-960a-b15d7d8f4230, 29, Finished, Available, Finished, False)

Updated patient dimension created successfully!
Total unique patients: 124150


SynapseWidget(Synapse.DataFrame, 83f45b0b-8985-4ecd-8482-0f3c3512362f)

#### 5.2 Create Payer Dimension

In [24]:
# ============================================================
# GOLD LAYER - STEP 2: CREATE DIM_PAYERS
# ============================================================

from pyspark.sql import functions as F

# 1. Load the original payer lookup table.
payers_df = spark.table("dbo_1.payers")

# 2. Select the payer ID and name.
#    Rename the columns to use consistent lowercase naming.
dim_payers = payers_df.select(
    F.col("Id").alias("payer_id"),
    F.col("NAME").alias("payer_name")
)

# 3. Remove duplicate payer IDs, if any.
dim_payers = dim_payers.dropDuplicates(["payer_id"])

# 4. Save the payer dimension as a Delta table.
dim_payers.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dim_payers")

# 5. Confirm successful creation.
print("Gold payer dimension created successfully!")
print("Total unique payers:", dim_payers.count())

# 6. Display the payer records.
display(dim_payers.orderBy("payer_name"))


StatementMeta(, a54bb5ee-e36c-4e77-960a-b15d7d8f4230, 30, Finished, Available, Finished, False)

Gold payer dimension created successfully!
Total unique payers: 10


SynapseWidget(Synapse.DataFrame, 676b7a3d-018f-44c4-9351-1024bf8eb016)

## <u>6 - Gold Layer Medication Fact Table</u>

#### 6.1 Create Medication Fact Table

In [25]:
# ============================================================
# GOLD LAYER - STEP 3: CREATE FACT_MEDICATIONS
# ============================================================

from pyspark.sql import functions as F

# 1. Load the enriched Silver medication table.
silver_df = spark.table("silver_medications_enriched")

# 2. Select the medication-level information needed for analysis.
#    Patient and payer IDs will connect this table to dimensions.
fact_medications = silver_df.select(
    "patient_id",
    "payer_id",
    "encounter_id",
    "medication_code",
    "medication_description",
    "medication_start",
    "medication_stop",
    "age_at_medication_start",
    "age_group",
    "base_cost",
    "total_cost",
    "payer_coverage",
    "calculated_uncovered_cost",
    "dispenses",
    "reason_code",
    "reason_description"
)

# 3. Save the medication fact table in Delta format.
#    Each row retains one original medication record.
fact_medications.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("fact_medications")

# 4. Confirm that the fact table was created.
print("Gold medication fact table created successfully!")

# 5. Count medication records.
print("Total medication records:", fact_medications.count())

# 6. Display the table structure.
fact_medications.printSchema()

# 7. Preview the first 10 records.
display(fact_medications.limit(10))


StatementMeta(, a54bb5ee-e36c-4e77-960a-b15d7d8f4230, 31, Finished, Available, Finished, False)

Gold medication fact table created successfully!
Total medication records: 4227723
root
 |-- patient_id: string (nullable = true)
 |-- payer_id: string (nullable = true)
 |-- encounter_id: string (nullable = true)
 |-- medication_code: integer (nullable = true)
 |-- medication_description: string (nullable = true)
 |-- medication_start: date (nullable = true)
 |-- medication_stop: date (nullable = true)
 |-- age_at_medication_start: integer (nullable = true)
 |-- age_group: string (nullable = true)
 |-- base_cost: double (nullable = true)
 |-- total_cost: double (nullable = true)
 |-- payer_coverage: double (nullable = true)
 |-- calculated_uncovered_cost: double (nullable = true)
 |-- dispenses: integer (nullable = true)
 |-- reason_code: long (nullable = true)
 |-- reason_description: string (nullable = true)



SynapseWidget(Synapse.DataFrame, 56220bbd-3de1-48b7-a346-f7f680cd8f52)

## <u>7 - Gold Layer Analytical Aggregations</u>

#### 7.1 Build Medication Utilization by Age Group
For "The Need" portion of the dashboard 

In [26]:
%%sql
-- ============================================================
-- GOLD LAYER - STEP 4: MEDICATION USAGE BY AGE GROUP
-- This will summarize medication activity by age group
-- ============================================================

CREATE OR REPLACE TABLE agg_medication_by_age
USING DELTA
AS

SELECT

    -- The age category assigned to each medication record.
    age_group,

    -- Count the number of medication records.
    COUNT(*) AS medication_records,

    -- Count each patient only once within the age group.
    COUNT(DISTINCT patient_id) AS unique_patients,

    -- Add up all medication dispensals.
    SUM(dispenses) AS total_dispenses,

    -- Calculate the average cost per medication record.
    ROUND(AVG(total_cost), 2) AS avg_medication_cost,

    -- Calculate medication records per represented patient.
    ROUND(
        COUNT(*) * 1.0 / COUNT(DISTINCT patient_id),
        2
    ) AS avg_medication_records_per_patient

FROM fact_medications

GROUP BY age_group;


StatementMeta(, a54bb5ee-e36c-4e77-960a-b15d7d8f4230, 32, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [27]:
%%sql
-- Display the age groups in their intended order to preview results.

SELECT *
FROM agg_medication_by_age

ORDER BY
    CASE age_group
        WHEN 'Under 18' THEN 1
        WHEN '18-29' THEN 2
        WHEN '30-44' THEN 3
        WHEN '45-59' THEN 4
        WHEN '60-74' THEN 5
        WHEN '75+' THEN 6
        ELSE 7
    END;


StatementMeta(, a54bb5ee-e36c-4e77-960a-b15d7d8f4230, 33, Finished, Available, Finished, False)

<Spark SQL result set with 6 rows and 6 fields>

### 7.2 - Build Insurance Coverage by Payer
For "The Insurance Gap" portion of the dashboard

In [28]:
%%sql
-- ============================================================
-- GOLD LAYER - STEP 5: INSURANCE COVERAGE BY PAYER
-- ============================================================

CREATE OR REPLACE TABLE agg_coverage_by_payer
USING DELTA
AS

SELECT

    -- Connect each payer ID to its readable insurance name.
    p.payer_id,
    p.payer_name,

    -- Count medication records associated with each payer.
    COUNT(*) AS medication_records,

    -- Count distinct patients associated with each payer.
    COUNT(DISTINCT f.patient_id) AS unique_patients,

    -- Calculate total medication costs.
    ROUND(SUM(f.total_cost), 2) AS total_medication_cost,

    -- Calculate total payer contributions.
    ROUND(SUM(f.payer_coverage), 2) AS total_payer_coverage,

    -- Calculate the uncovered portion of medication costs.
    ROUND(SUM(f.calculated_uncovered_cost), 2)
        AS total_calculated_uncovered_cost,

    -- Calculate weighted coverage percentage.
    ROUND(
        SUM(f.payer_coverage) * 100.0 /
        NULLIF(SUM(f.total_cost), 0),
        2
    ) AS coverage_percentage

FROM fact_medications f

-- Join the payer dimension to retrieve readable payer names.
INNER JOIN dim_payers p
    ON f.payer_id = p.payer_id

GROUP BY
    p.payer_id,
    p.payer_name;


StatementMeta(, a54bb5ee-e36c-4e77-960a-b15d7d8f4230, 34, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [29]:
%%sql
-- Preview the results
SELECT *
FROM agg_coverage_by_payer
ORDER BY coverage_percentage DESC;


StatementMeta(, a54bb5ee-e36c-4e77-960a-b15d7d8f4230, 35, Finished, Available, Finished, False)

<Spark SQL result set with 10 rows and 8 fields>

#### 7.3 Build Insurance Coverage by Age Group
For "The Insurance Gap" portion of the dashbaord

In [30]:
%%sql
-- Create an age-level summary of medication costs and payer contributions.
CREATE OR REPLACE TABLE agg_coverage_by_age
USING DELTA
AS
SELECT
    -- Group results by the patient's age when the medication began.
    age_group,

    -- Count medication records and distinct patients in each age group.
    COUNT(*) AS medication_records,
    COUNT(DISTINCT patient_id) AS unique_patients,

    -- Calculate total medication costs and payer contributions.
    ROUND(SUM(total_cost), 2) AS total_medication_cost,
    ROUND(SUM(payer_coverage), 2) AS total_payer_coverage,

    -- Calculate the remaining cost after payer contributions.
    ROUND(SUM(calculated_uncovered_cost), 2)
        AS total_calculated_uncovered_cost,

    -- Calculate weighted coverage across all medication records.
    ROUND(
        SUM(payer_coverage) * 100.0 /
        NULLIF(SUM(total_cost), 0),
        2
    ) AS coverage_percentage,

    -- Calculate average calculated uncovered cost per unique patient.
    ROUND(
        SUM(calculated_uncovered_cost) /
        NULLIF(COUNT(DISTINCT patient_id), 0),
        2
    ) AS avg_calculated_uncovered_per_patient

FROM fact_medications

-- Summarize the data separately for each age group.
GROUP BY age_group;

StatementMeta(, a54bb5ee-e36c-4e77-960a-b15d7d8f4230, 36, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [31]:
%%sql
-- Preview the result
-- Display age groups in their natural order.
SELECT *
FROM agg_coverage_by_age
ORDER BY
    CASE age_group
        WHEN 'Under 18' THEN 1
        WHEN '18-29' THEN 2
        WHEN '30-44' THEN 3
        WHEN '45-59' THEN 4
        WHEN '60-74' THEN 5
        WHEN '75+' THEN 6
        ELSE 7
    END;

StatementMeta(, a54bb5ee-e36c-4e77-960a-b15d7d8f4230, 37, Finished, Available, Finished, False)

<Spark SQL result set with 6 rows and 8 fields>

#### 7.4 - Build Patient Medication Burden Summary
For "The Burden" portion of the dashboard

In [33]:
%%sql
-- Create a patient-level summary of medication utilization and costs.
CREATE OR REPLACE TABLE agg_patient_medication_burden
USING DELTA
AS
SELECT
    -- Identify each patient.
    patient_id,

    -- Count all medication records associated with the patient.
    COUNT(*) AS medication_records,

    -- Count distinct medication types using their medication codes.
    COUNT(DISTINCT medication_code) AS distinct_medication_count,

    -- Calculate the patient's cumulative medication costs.
    ROUND(SUM(total_cost), 2) AS total_medication_cost,

    -- Calculate the total amount contributed by payers.
    ROUND(SUM(payer_coverage), 2) AS total_payer_coverage,

    -- Calculate the remaining cost after payer contributions.
    ROUND(SUM(calculated_uncovered_cost), 2)
        AS total_calculated_uncovered_cost

FROM fact_medications

-- Group all medication records belonging to the same patient.
GROUP BY patient_id;

StatementMeta(, a54bb5ee-e36c-4e77-960a-b15d7d8f4230, 39, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [34]:
%%sql
-- Check the number of patients represented in the new table to preview results
SELECT
    COUNT(*) AS total_patients,

    -- Check the smallest and largest distinct medication counts.
    MIN(distinct_medication_count) AS min_medication_types,
    MAX(distinct_medication_count) AS max_medication_types,

    -- Check whether any patient has a missing medication count.
    SUM(
        CASE
            WHEN distinct_medication_count IS NULL THEN 1
            ELSE 0
        END
    ) AS missing_medication_counts

FROM agg_patient_medication_burden;

StatementMeta(, a54bb5ee-e36c-4e77-960a-b15d7d8f4230, 40, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 4 fields>

#### 7.5 Build Medication Burden Aggregation 

In [35]:
%%sql
-- Create the final Gold aggregation for the financial burden analysis.
CREATE OR REPLACE TABLE agg_burden_by_med_count
USING DELTA
AS
SELECT
    -- Assign each patient to a medication-count category.
    CASE
        WHEN distinct_medication_count BETWEEN 1 AND 2
            THEN '1-2'
        WHEN distinct_medication_count BETWEEN 3 AND 4
            THEN '3-4'
        WHEN distinct_medication_count BETWEEN 5 AND 6
            THEN '5-6'
        WHEN distinct_medication_count BETWEEN 7 AND 8
            THEN '7-8'
        WHEN distinct_medication_count >= 9
            THEN '9+'
        ELSE 'Unknown'
    END AS medication_count_group,

    -- Count the number of patients in each category.
    COUNT(*) AS unique_patients,

    -- Calculate the average number of distinct medication types.
    ROUND(AVG(distinct_medication_count), 2)
        AS avg_distinct_medication_count,

    -- Calculate cumulative medication costs across patients.
    ROUND(SUM(total_medication_cost), 2)
        AS total_medication_cost,

    -- Calculate total payer contributions.
    ROUND(SUM(total_payer_coverage), 2)
        AS total_payer_coverage,

    -- Calculate the remaining costs after payer contributions.
    ROUND(SUM(total_calculated_uncovered_cost), 2)
        AS total_calculated_uncovered_cost,

    -- Calculate average calculated uncovered cost per patient.
    ROUND(
        SUM(total_calculated_uncovered_cost) / COUNT(*),
        2
    ) AS avg_calculated_uncovered_per_patient,

    -- Calculate weighted payer coverage for each group.
    ROUND(
        SUM(total_payer_coverage) * 100.0 /
        NULLIF(SUM(total_medication_cost), 0),
        2
    ) AS coverage_percentage

FROM agg_patient_medication_burden

-- Combine patients who belong to the same medication-count group.
GROUP BY
    CASE
        WHEN distinct_medication_count BETWEEN 1 AND 2 THEN '1-2'
        WHEN distinct_medication_count BETWEEN 3 AND 4 THEN '3-4'
        WHEN distinct_medication_count BETWEEN 5 AND 6 THEN '5-6'
        WHEN distinct_medication_count BETWEEN 7 AND 8 THEN '7-8'
        WHEN distinct_medication_count >= 9 THEN '9+'
        ELSE 'Unknown'
    END;

StatementMeta(, a54bb5ee-e36c-4e77-960a-b15d7d8f4230, 41, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [36]:
%%sql
-- Display the medication-count categories in their intended order to preview results
SELECT *
FROM agg_burden_by_med_count
ORDER BY
    CASE medication_count_group
        WHEN '1-2' THEN 1
        WHEN '3-4' THEN 2
        WHEN '5-6' THEN 3
        WHEN '7-8' THEN 4
        WHEN '9+' THEN 5
        ELSE 6
    END;

StatementMeta(, a54bb5ee-e36c-4e77-960a-b15d7d8f4230, 42, Finished, Available, Finished, False)

<Spark SQL result set with 5 rows and 8 fields>

## <u>8 - Final Gold Layer Validation</u>

In [37]:
%%sql
-- Compare the number of patients across our fact, patient-level,
-- and medication-count aggregation tables.
SELECT
    (SELECT COUNT(DISTINCT patient_id)
     FROM fact_medications) AS fact_unique_patients,

    (SELECT COUNT(*)
     FROM agg_patient_medication_burden) AS patient_summary_rows,

    (SELECT SUM(unique_patients)
     FROM agg_burden_by_med_count) AS patients_in_medication_groups,

    -- Check whether any patients were assigned to an unexpected group.
    (SELECT COUNT(*)
     FROM agg_burden_by_med_count
     WHERE medication_count_group = 'Unknown') AS unknown_groups,

    -- Compare total medication costs from the fact table
    -- against the summarized medication-count groups.
    (SELECT ROUND(SUM(total_cost), 2)
     FROM fact_medications) AS fact_total_medication_cost,

    (SELECT ROUND(SUM(total_medication_cost), 2)
     FROM agg_burden_by_med_count) AS grouped_total_medication_cost,

    -- Compare calculated uncovered costs.
    (SELECT ROUND(SUM(calculated_uncovered_cost), 2)
     FROM fact_medications) AS fact_total_uncovered_cost,

    (SELECT ROUND(SUM(total_calculated_uncovered_cost), 2)
     FROM agg_burden_by_med_count) AS grouped_total_uncovered_cost;

StatementMeta(, a54bb5ee-e36c-4e77-960a-b15d7d8f4230, 43, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 8 fields>

## <u>9 - Medication Coverage by Age Group and Payer</u>

Using the Gold medication and payer tables, calculate the share of medication costs covered by each payer within each age group at medication start. Coverage is total recorded payer contributions divided by total medication costs. A group with zero total medication cost has a null coverage proportion.

In [ ]:
"""Query weighted medication coverage by age group and payer in Spark."""


def query_coverage_by_age_and_payer(spark):
    """Return one row per age-at-medication-start group and payer ID."""
    return spark.sql("""
        WITH coverage_totals AS (
            SELECT
                age_group,
                payer_id,
                COUNT(*) AS medication_records,
                SUM(total_cost) AS total_medication_cost,
                SUM(payer_coverage) AS total_payer_coverage
            FROM fact_medications
            GROUP BY age_group, payer_id
        )
        SELECT
            totals.age_group,
            totals.payer_id,
            payers.payer_name,
            totals.medication_records,
            totals.total_medication_cost,
            totals.total_payer_coverage,
            totals.total_payer_coverage / NULLIF(totals.total_medication_cost, 0)
                AS coverage_proportion
        FROM coverage_totals AS totals
        LEFT JOIN dim_payers AS payers
            ON totals.payer_id = payers.payer_id
        ORDER BY
            CASE totals.age_group
                WHEN 'Under 18' THEN 1
                WHEN '18-29' THEN 2
                WHEN '30-44' THEN 3
                WHEN '45-59' THEN 4
                WHEN '60-74' THEN 5
                WHEN '75+' THEN 6
                ELSE 7
            END,
            totals.age_group,
            payers.payer_name,
            totals.payer_id
    """)


coverage_by_age_and_payer = query_coverage_by_age_and_payer(spark)
display(coverage_by_age_and_payer)
